In [2]:
import pandas as pd
from sklearn.metrics import mean_squared_error

random_state = 42

In [3]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv"
rating_df = pd.read_csv(rating_url)
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,3.0
1,1342067,CL0101EN,3.0
2,1990814,ML0120ENv3,3.0
3,380098,BD0211EN,3.0
4,779563,DS0101EN,3.0


In [4]:
num_users = len(rating_df['user'].unique())
num_items = len(rating_df['item'].unique())
print(f"There are total {num_users} users and {num_items} items.")

There are total 33901 users and 126 items.


In [5]:
def process_dataset(raw_data):
    encoded_data = raw_data.copy()
    user_list = encoded_data['user'].unique().tolist()
    user_id2idx_dict = {x: i for i, x in enumerate(user_list)}
    user_idx2id_dict = dict(enumerate(user_list))

    course_list = encoded_data['item'].unique().tolist()
    course_id2idx_dict = {x: i for i, x in enumerate(course_list)}
    course_idx2id_dict = dict(enumerate(course_list))

    encoded_data['user'] = encoded_data['user'].map(user_id2idx_dict)
    encoded_data['item'] = encoded_data['item'].map(course_id2idx_dict)
    encoded_data['rating'] = encoded_data['rating'].astype(int)

    return encoded_data, user_idx2id_dict, course_idx2id_dict

In [6]:
encoded_data, user_idx2id_dict, course_idx2id_dict = process_dataset(rating_df)

In [7]:
encoded_data.head()

,user,item,rating
0,0,0,3
1,1,1,3
2,2,2,3
3,3,3,3
4,4,4,3


In [9]:
def generate_train_test_datasets(dataset):
    min_rating = min(dataset['rating'])

    dataset = dataset.sample(frac=1, random_state=random_state)
    x = dataset[['user', 'item']].values
    y = dataset['rating'].apply(lambda x: x - min_rating).values

    train_indices = int(0.8 * dataset.shape[0])
    test_indices = int(0.9 * dataset.shape[0])

    x_train, x_val, x_test, y_train, y_val, y_test = (
        x[:train_indices],
        x[train_indices:test_indices],
        x[test_indices:],
        y[:train_indices],
        y[train_indices:test_indices],
        y[test_indices:]
    )
    return x_train, x_val, x_test, y_train, y_val, y_test

In [10]:
x_train, x_val, x_test, y_train, y_val, y_test = generate_train_test_datasets(encoded_data)

In [11]:
user_indices = x_train[:, 0]
user_indices

array([ 8376,  7659, 10717, ...,  3409, 28761,  4973], shape=(186644,))

In [12]:
item_indices = x_train[:, 0]
item_indices

array([ 8376,  7659, 10717, ...,  3409, 28761,  4973], shape=(186644,))

In [13]:
y_train

array([1, 1, 1, ..., 1, 0, 1], shape=(186644,))

In [14]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x_train_tensor = torch.tensor(x_train, dtype=torch.long)
x_val_tensor = torch.tensor(x_val, dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

In [15]:
from torch import nn


class RecommenderNet(nn.Module):
    def __init__(self, num_users, num_items, num_classes, embedding_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)

        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.item_bias = nn.Embedding(num_items, 1)

        self.regressor = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        nn.init.kaiming_normal_(self.user_embedding.weight)
        nn.init.kaiming_normal_(self.item_embedding.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, inputs):
        users, items = inputs[:, 0], inputs[:, 1]
        user_vector = self.user_embedding(users)
        item_vector = self.item_embedding(items)

        user_bias = self.user_bias(users)
        item_bias = self.item_bias(items)

        x = torch.cat([user_vector, item_vector], dim=1)
        logits = self.regressor(x)
        logits = logits + user_bias + item_bias

        return logits

In [18]:
model = RecommenderNet(num_users, num_items, len(rating_df['rating'].unique()))

In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
criterion = nn.CrossEntropyLoss()

In [21]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(train_loader.dataset)

In [20]:
def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * x_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    avg_loss = total_loss / len(val_loader.dataset)
    accuracy = correct / total
    return avg_loss, accuracy

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
n_epochs = 10
for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, accuracy = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.4f}")

Epoch 1, Train Loss: 0.2166, Val Loss: 0.1203, Val Acc: 0.9607
Epoch 2, Train Loss: 0.1084, Val Loss: 0.1004, Val Acc: 0.9672
Epoch 3, Train Loss: 0.0925, Val Loss: 0.0861, Val Acc: 0.9711
Epoch 4, Train Loss: 0.0702, Val Loss: 0.0681, Val Acc: 0.9758
Epoch 5, Train Loss: 0.0450, Val Loss: 0.0528, Val Acc: 0.9802
Epoch 6, Train Loss: 0.0268, Val Loss: 0.0427, Val Acc: 0.9852
Epoch 7, Train Loss: 0.0160, Val Loss: 0.0368, Val Acc: 0.9867
Epoch 8, Train Loss: 0.0102, Val Loss: 0.0334, Val Acc: 0.9894
Epoch 9, Train Loss: 0.0071, Val Loss: 0.0323, Val Acc: 0.9893
Epoch 10, Train Loss: 0.0053, Val Loss: 0.0316, Val Acc: 0.9889


In [27]:
import numpy as np
from sklearn.metrics import mean_squared_error

def test(model, test_loader, device, min_rating):
    model.eval()
    all_true = []
    all_pred = []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            preds = torch.argmax(logits, dim=1)
            true_ratings = y_batch.cpu().numpy() + min_rating
            pred_ratings = preds.cpu().numpy() + min_rating

            all_true.extend(true_ratings.tolist())
            all_pred.extend(pred_ratings.tolist())
    rmse = np.sqrt(mean_squared_error(all_true, all_pred))
    accuracy = np.mean(np.array(all_true) == np.array(all_pred))
    return rmse, accuracy

In [28]:
test_rmse, test_acc = test(model, test_loader, device, min(rating_df['rating']))
print(f"Test RMSE: {test_rmse:.4f}, Test Accuracy: {test_acc:.4f}")

Test RMSE: 0.1064, Test Accuracy: 0.9887
